# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, overview, and analyze the FAIR² dataset ([source](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)) using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library, referencing all structures by their `@id` and following a reproducible, semantic workflow.

### Dataset Source
The dataset schema is provided via a Croissant JSON-LD document at:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

We first load the dataset metadata from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset source URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Metadata: name and description (attribute access only!)
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")

## 2. Data Overview

Let's overview the record sets, their fields, columns, and their `@id` in the dataset. This helps us understand how to access and reference the data in subsequent steps.

In [ ]:
# List all record sets and their fields with `@id`

for record_set in dataset.record_sets:
    print(f"RecordSet Name: {record_set.name}\n  @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id})")
    print()

## 3. Data Extraction

Here, we load all records for each record set (referenced **by `@id`**) into in-memory DataFrames using Pandas. We will display column names and a sample for each available record set.

*Note: For this demonstration, if the schema contains more than one record set, we'll extract all; please adjust `record_sets` as necessary based on actual schema content obtained above.*

In [ ]:
# Gather all available record set @id's
record_sets = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets:
    print(f"\n----\nLoading records from RecordSet (@id): {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"> Columns: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate typical EDA workflows referencing columns **by their field `@id`**; filtering, normalizing, grouping, and handling records semantically. *(Please fill in valid field `@id`s from the overview above.)*

In [ ]:
# Select the main RecordSet to analyze (first from overview)
main_recordset_id = record_sets[0]
df = dataframes[main_recordset_id]

# Identify numeric and categorical field @id's (please pick from previous overview output)
# For demonstration, let's use 'Age' and 'Sex' -- replace with the actual field @id
numeric_field = [col for col in df.columns if 'age' in col.lower() or 'Age' in col][0] if any('age' in col.lower() for col in df.columns) else df.columns[0]
group_field   = [col for col in df.columns if any(word in col.lower() for word in ['sex', 'gender'])][0] if any(word in col.lower() for col in df.columns for word in ['sex', 'gender']) else df.columns[-1]

print(f"Numeric field selected (@id): {numeric_field}")
print(f"Group field selected   (@id): {group_field}")

# Filter on the numeric field (age > 50 as an example threshold; adjust as necessary)
try:
    threshold = 50
    filtered_df = df[df[numeric_field].astype(float) > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) /
        filtered_df[numeric_field].astype(float).std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by the group_field if present, and show means
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (mean of numeric fields):")
        display(grouped_df)
except Exception as e:
    print(f"No suitable numeric values for field {numeric_field} or grouping, or cast error: {e}")

## 5. Visualization

We can visualize distributions or relationships between fields using [matplotlib](https://matplotlib.org/) and [seaborn](https://seaborn.pydata.org/), referencing columns by their field `@id`.

*Example: Plot distribution of the selected numeric field or a boxplot grouped by group field.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
if numeric_field in df.columns:
    sns.histplot(df[numeric_field].astype(float), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

if group_field in df.columns and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion

+ Successfully loaded and explored the FAIR² dataset package using `mlcroissant`.
+ Data elements (`recordSets`, `fields`, and columns) were referenced and manipulated using their `@id` identifiers.
+ Example workflow included metadata inspection, record set extraction, numeric filtering, normalization, and simple groupwise summaries.
+ The notebook can be easily extended for more advanced EDA or machine learning workflows—just continue referencing schema entities by their `@id`!

For additional details, see the [official mlcroissant documentation](https://mlcommons.github.io/croissant/).